# Merge attack and normal datasets

## Loading and cleaning

In [14]:
import pandas as pd
import numpy as np
normal_train=pd.read_csv("data/normal_train/normal_2_3_4.csv")
normal_test=pd.read_csv("data/normal_test/normal_5_6_7.csv")
attack_train=pd.read_csv("data/dollar_char_train/dollar_char_train_labeled.csv")
attack_test=pd.read_csv("data/dollar_char_test/dollar_char_test_labeled.csv")

/tmp/ipykernel_2264304/1206338882.py:3: DtypeWarning: Columns (0: client_fingerprint, 1: server_fingerprint) have mixed types. Specify dtype option on import or set low_memory=False.
  normal_train=pd.read_csv("data/normal_train/normal_2_3_4.csv")
/tmp/ipykernel_2264304/1206338882.py:4: DtypeWarning: Columns (0: client_fingerprint, 1: server_fingerprint) have mixed types. Specify dtype option on import or set low_memory=False.
  normal_test=pd.read_csv("data/normal_test/normal_5_6_7.csv")
/tmp/ipykernel_2264304/1206338882.py:5: DtypeWarning: Columns (0: requested_server_name, 1: client_fingerprint, 2: server_fingerprint) have mixed types. Specify dtype option on import or set low_memory=False.
  attack_train=pd.read_csv("data/dollar_char_train/dollar_char_train_labeled.csv")
/tmp/ipykernel_2264304/1206338882.py:6: DtypeWarning: Columns (0: requested_server_name, 1: client_fingerprint, 2: server_fingerprint) have mixed types. Specify dtype option on import or set low_memory=False.
  att

In [15]:
to_be_dropped=[col for col in normal_train.columns if "Unnamed" in col]
normal_train=normal_train.drop(to_be_dropped, axis=1)
normal_test=normal_test.drop(to_be_dropped, axis=1)

In [16]:
normal_train.info()

<class 'pandas.DataFrame'>
RangeIndex: 9441 entries, 0 to 9440
Data columns (total 86 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   id                            9441 non-null   int64  
 1   expiration_id                 9441 non-null   int64  
 2   src_ip                        9441 non-null   str    
 3   src_mac                       9441 non-null   str    
 4   src_oui                       9441 non-null   str    
 5   src_port                      9441 non-null   int64  
 6   dst_ip                        9441 non-null   str    
 7   dst_mac                       9441 non-null   str    
 8   dst_oui                       9441 non-null   str    
 9   dst_port                      9441 non-null   int64  
 10  protocol                      9441 non-null   int64  
 11  ip_version                    9441 non-null   int64  
 12  vlan_id                       9441 non-null   int64  
 13  tunnel_id     

In [17]:
normal_test.info()

<class 'pandas.DataFrame'>
RangeIndex: 9411 entries, 0 to 9410
Data columns (total 86 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   id                            9411 non-null   int64  
 1   expiration_id                 9411 non-null   int64  
 2   src_ip                        9411 non-null   str    
 3   src_mac                       9411 non-null   str    
 4   src_oui                       9411 non-null   str    
 5   src_port                      9411 non-null   int64  
 6   dst_ip                        9411 non-null   str    
 7   dst_mac                       9411 non-null   str    
 8   dst_oui                       9411 non-null   str    
 9   dst_port                      9411 non-null   int64  
 10  protocol                      9411 non-null   int64  
 11  ip_version                    9411 non-null   int64  
 12  vlan_id                       9411 non-null   int64  
 13  tunnel_id     

In [18]:
to_be_dropped=["phase_idx", "phase_name", "phase_number"] + [col for col in attack_train.columns if "Unnamed" in col]
print(to_be_dropped)

['phase_idx', 'phase_name', 'phase_number', 'Unnamed: 0']


In [19]:
attack_train=attack_train.drop(to_be_dropped, axis=1)
attack_test=attack_test.drop(to_be_dropped, axis=1)

In [20]:
attack_train.info()

<class 'pandas.DataFrame'>
RangeIndex: 195044 entries, 0 to 195043
Data columns (total 89 columns):
 #   Column                        Non-Null Count   Dtype  
---  ------                        --------------   -----  
 0   id                            195044 non-null  int64  
 1   expiration_id                 195044 non-null  int64  
 2   src_ip                        195044 non-null  str    
 3   src_mac                       195044 non-null  str    
 4   src_oui                       195044 non-null  str    
 5   src_port                      195044 non-null  int64  
 6   dst_ip                        195044 non-null  str    
 7   dst_mac                       195044 non-null  str    
 8   dst_oui                       195044 non-null  str    
 9   dst_port                      195044 non-null  int64  
 10  protocol                      195044 non-null  int64  
 11  ip_version                    195044 non-null  int64  
 12  vlan_id                       195044 non-null  int64  


In [21]:
attack_test.info()

<class 'pandas.DataFrame'>
RangeIndex: 110276 entries, 0 to 110275
Data columns (total 89 columns):
 #   Column                        Non-Null Count   Dtype  
---  ------                        --------------   -----  
 0   id                            110276 non-null  int64  
 1   expiration_id                 110276 non-null  int64  
 2   src_ip                        110276 non-null  str    
 3   src_mac                       110276 non-null  str    
 4   src_oui                       110276 non-null  str    
 5   src_port                      110276 non-null  int64  
 6   dst_ip                        110276 non-null  str    
 7   dst_mac                       110276 non-null  str    
 8   dst_oui                       110276 non-null  str    
 9   dst_port                      110276 non-null  int64  
 10  protocol                      110276 non-null  int64  
 11  ip_version                    110276 non-null  int64  
 12  vlan_id                       110276 non-null  int64  


## Convert to IPv4

In [22]:
mappings_attack={}

for _, row in attack_train.iterrows():
    src_mac=row["src_mac"]
    dst_mac=row["dst_mac"]
    src_ip=row["src_ip"]
    dst_ip=row["dst_ip"]
    if ":" not in src_ip:
        mappings_attack[src_mac]=src_ip
    if ":" not in dst_ip:
        mappings_attack[dst_mac]=dst_ip

for _, row in attack_test.iterrows():
    src_mac=row["src_mac"]
    dst_mac=row["dst_mac"]
    src_ip=row["src_ip"]
    dst_ip=row["dst_ip"]
    if ":" not in src_ip:
        mappings_attack[src_mac]=src_ip
    if ":" not in dst_ip:
        mappings_attack[dst_mac]=dst_ip        
        

In [23]:
print(mappings_attack)

{'12:34:7e:5d:bf:ed': '10.0.0.2', '12:34:c6:7d:c4:17': '10.0.0.4', '12:34:2f:04:c4:76': '10.0.0.6', '12:34:19:14:0c:5b': '10.0.0.8', '12:34:cd:78:ac:c7': '10.0.0.5', '12:34:fb:02:73:4a': '10.0.0.7', '12:34:a2:17:83:65': '10.0.0.13', '12:34:9b:aa:04:c0': '10.0.0.12', '12:34:88:d1:ed:07': '10.0.0.11', '12:34:d2:15:14:d2': '10.0.0.9', '12:34:13:7c:72:38': '10.0.0.10', '12:34:9e:6b:c3:97': '10.0.0.16', '12:34:f3:ef:1f:0e': '10.0.0.23', '12:34:66:23:dc:bd': '10.0.0.22', '12:34:7c:fe:43:ea': '10.0.0.21', '12:34:ce:17:98:03': '10.0.0.1', '12:34:2f:33:35:d0': '10.0.0.19', '12:34:68:e8:37:59': '10.0.0.18', '12:34:93:88:12:84': '10.0.0.17', '12:34:3c:ae:94:28': '10.0.0.20', '12:34:08:7a:17:c7': '10.0.0.14', '12:34:dd:19:27:75': '10.0.0.15', '12:34:01:19:20:f8': '10.0.0.2', '12:34:76:cf:4c:7e': '10.0.0.5', '12:34:ab:ca:6b:5f': '10.0.0.4', '12:34:d6:2e:4c:74': '10.0.0.6', '12:34:26:0e:c5:a6': '10.0.0.7', '12:34:80:b8:c1:2f': '10.0.0.8', '12:34:32:bb:6b:ea': '10.0.0.9', '12:34:02:8e:4e:30': '10.0.0

In [24]:
mappings_normal={}

for _, row in normal_train.iterrows():
    src_mac=row["src_mac"]
    dst_mac=row["dst_mac"]
    src_ip=row["src_ip"]
    dst_ip=row["dst_ip"]
    if ":" not in src_ip:
        mappings_normal[src_mac]=src_ip
    if ":" not in dst_ip:
        mappings_normal[dst_mac]=dst_ip

for _, row in normal_test.iterrows():
    src_mac=row["src_mac"]
    dst_mac=row["dst_mac"]
    src_ip=row["src_ip"]
    dst_ip=row["dst_ip"]
    if ":" not in src_ip:
        mappings_normal[src_mac]=src_ip
    if ":" not in dst_ip:
        mappings_normal[dst_mac]=dst_ip        
        

In [25]:
print(mappings_normal)

{'12:34:73:b3:6d:24': '10.0.0.22', '12:34:a5:15:3c:3e': '10.0.0.1', '12:34:d4:3b:81:a1': '10.0.0.23', '12:34:6f:ad:cc:cd': '10.0.0.4', '12:34:d7:7e:03:0b': '10.0.0.5', '12:34:7f:bb:40:6c': '10.0.0.6', '12:34:4b:9c:8f:9a': '10.0.0.7', '12:34:78:e0:c0:a0': '10.0.0.8', '12:34:1d:ee:d8:f2': '10.0.0.9', '12:34:23:73:75:33': '10.0.0.15', '12:34:0d:f0:9c:36': '10.0.0.13', '12:34:a1:83:ca:33': '10.0.0.11', '12:34:8f:d5:2b:9a': '10.0.0.10', '12:34:11:9f:d4:8e': '10.0.0.12', '12:34:2e:58:19:93': '10.0.0.14', '12:34:9f:d5:63:9f': '10.0.0.16', '12:34:32:ca:d7:20': '10.0.0.17', '12:34:94:48:0a:74': '10.0.0.19', '12:34:a5:12:d2:5f': '10.0.0.2', '12:34:c1:b5:04:96': '10.0.0.18', '12:34:25:3a:5e:08': '10.0.0.21', '12:34:8f:a1:bd:44': '10.0.0.20', '12:34:e6:e8:7c:cc': '10.0.0.7', '12:34:4c:3c:6b:02': '10.0.0.1', '12:34:c3:58:c9:90': '10.0.0.8', '12:34:da:e9:f7:5f': '10.0.0.4', '12:34:fa:fb:5e:7d': '10.0.0.5', '12:34:c0:72:98:90': '10.0.0.9', '12:34:50:6c:ce:be': '10.0.0.12', '12:34:98:73:9b:79': '10.0.

In [26]:
print(mappings_attack.keys() & mappings_normal.keys())

set()


In [27]:
mappings = mappings_attack | mappings_normal

In [28]:
print(mappings)

{'12:34:7e:5d:bf:ed': '10.0.0.2', '12:34:c6:7d:c4:17': '10.0.0.4', '12:34:2f:04:c4:76': '10.0.0.6', '12:34:19:14:0c:5b': '10.0.0.8', '12:34:cd:78:ac:c7': '10.0.0.5', '12:34:fb:02:73:4a': '10.0.0.7', '12:34:a2:17:83:65': '10.0.0.13', '12:34:9b:aa:04:c0': '10.0.0.12', '12:34:88:d1:ed:07': '10.0.0.11', '12:34:d2:15:14:d2': '10.0.0.9', '12:34:13:7c:72:38': '10.0.0.10', '12:34:9e:6b:c3:97': '10.0.0.16', '12:34:f3:ef:1f:0e': '10.0.0.23', '12:34:66:23:dc:bd': '10.0.0.22', '12:34:7c:fe:43:ea': '10.0.0.21', '12:34:ce:17:98:03': '10.0.0.1', '12:34:2f:33:35:d0': '10.0.0.19', '12:34:68:e8:37:59': '10.0.0.18', '12:34:93:88:12:84': '10.0.0.17', '12:34:3c:ae:94:28': '10.0.0.20', '12:34:08:7a:17:c7': '10.0.0.14', '12:34:dd:19:27:75': '10.0.0.15', '12:34:01:19:20:f8': '10.0.0.2', '12:34:76:cf:4c:7e': '10.0.0.5', '12:34:ab:ca:6b:5f': '10.0.0.4', '12:34:d6:2e:4c:74': '10.0.0.6', '12:34:26:0e:c5:a6': '10.0.0.7', '12:34:80:b8:c1:2f': '10.0.0.8', '12:34:32:bb:6b:ea': '10.0.0.9', '12:34:02:8e:4e:30': '10.0.0

In [29]:
normal_train["src_mac"].unique()

<StringArray>
['96:a5:fa:53:10:d0', 'fa:be:65:a8:61:93', '52:1f:a5:ce:9c:13',
 '12:34:6f:ad:cc:cd', '42:20:90:67:1a:fb', '62:6a:e0:b0:96:9f',
 '12:34:11:9f:d4:8e', '12:34:a5:12:d2:5f', '12:34:4b:9c:8f:9a',
 '06:24:39:86:47:fc', 'aa:10:28:4e:19:17', '02:d3:3f:1c:a0:0f',
 '26:99:57:7a:6e:93', '62:65:53:3c:4a:0e', '1e:5d:d7:80:d0:a6',
 'd6:fe:20:e8:71:40', '2e:75:07:80:63:05', 'aa:ca:9e:5f:37:b4',
 '72:83:74:0e:b8:b9', '66:6f:76:42:cf:b6', 'fe:12:71:3a:e8:64',
 'b2:ac:33:f9:4d:5a', '2a:22:e2:30:71:52', '3a:9c:c1:a8:17:39',
 '12:34:d4:3b:81:a1', '12:34:94:48:0a:74', '66:a7:dc:28:9d:48',
 '32:44:2c:c6:70:cd', '12:34:1d:ee:d8:f2', '12:34:9f:d5:63:9f',
 '12:34:2e:58:19:93', '12:34:a1:83:ca:33', '12:34:23:73:75:33',
 '12:34:78:e0:c0:a0', '12:34:44:30:94:c1', '3a:f9:b2:f6:55:18',
 '7a:85:dc:af:82:cc', '16:84:58:c5:96:4e', '1a:e0:de:ee:df:90',
 'ca:a9:a8:7e:a6:16', 'c2:6c:f3:35:65:f2', 'fe:73:58:53:d6:db',
 'c6:c5:16:b7:17:bb', '56:e7:f5:54:62:10', '6a:e1:34:d6:c6:eb',
 'b2:c2:a7:f0:40:33', '32:

In [30]:
normal_train["dst_mac"].unique()

<StringArray>
['33:33:00:00:00:16', '33:33:ff:a8:61:93', '33:33:ff:ce:9c:13',
 '33:33:00:00:00:02', '33:33:00:00:00:fb', '33:33:ff:b0:96:9f',
 '33:33:00:00:00:01', '12:34:a5:15:3c:3e', '33:33:ff:af:82:cc',
 '33:33:ff:f6:55:18', '33:33:ff:7e:a6:16', '33:33:ff:c5:96:4e',
 '33:33:ff:ee:df:90', '33:33:ff:35:65:f2', '33:33:ff:53:d6:db',
 '33:33:ff:54:62:10', '33:33:ff:53:10:d0', '33:33:ff:b7:17:bb',
 '33:33:ff:d6:c6:eb', '33:33:ff:78:c5:d1', '33:33:ff:f0:40:33',
 '33:33:ff:e7:90:05', '33:33:ff:32:5e:9f', '33:33:ff:40:02:8d',
 '33:33:ff:09:0c:2c', '33:33:ff:bf:a7:31', '33:33:ff:8e:fc:c6',
 '33:33:ff:51:37:08', '12:34:0d:f0:9c:36', '12:34:73:b3:6d:24',
 '12:34:11:9f:d4:8e', '12:34:32:ca:d7:20', '12:34:9f:d5:63:9f',
 '12:34:d4:3b:81:a1', '12:34:7f:bb:40:6c', '12:34:8f:d5:2b:9a',
 '12:34:c1:b5:04:96', '12:34:6f:ad:cc:cd', '12:34:a1:83:ca:33',
 '12:34:d7:7e:03:0b', '12:34:2e:58:19:93', '12:34:25:3a:5e:08',
 '12:34:78:e0:c0:a0', '12:34:1d:ee:d8:f2', '12:34:8f:a1:bd:44',
 '12:34:4b:9c:8f:9a', '12:

In [31]:
attack_train["src_mac"].unique()

<StringArray>
['12:34:7e:5d:bf:ed', '12:34:93:88:12:84', '12:34:f3:ef:1f:0e',
 '12:34:2f:04:c4:76', '12:34:fb:02:73:4a', '12:34:c6:7d:c4:17',
 '12:34:13:7c:72:38', '12:34:88:d1:ed:07', '12:34:9b:aa:04:c0',
 'd6:c6:da:6a:f2:23', 'f2:6f:34:cc:e6:80', '12:1c:63:b7:2e:27',
 '8e:6c:4b:ed:88:52', '3a:87:65:11:e9:ef', 'de:f4:23:ef:2e:b2',
 '1a:fa:bb:36:54:a1', '86:c1:8f:5a:11:6c', '4a:9b:0a:0e:43:98',
 'fa:dc:27:d5:18:ce', '82:8d:f9:07:6e:25', 'fa:31:32:9a:f0:74',
 '6a:db:6d:35:cf:81', '5a:dd:2b:c5:7c:67', '36:14:bb:67:13:83',
 'ce:14:54:91:ef:63', '22:45:06:b1:e3:19', '3a:cb:8e:92:9e:dc',
 'f6:9a:9c:18:1b:34', 'ce:67:13:91:3d:ad', '92:50:7b:c4:b4:c5',
 'aa:8d:5c:96:34:29', 'ee:00:f9:78:f2:ab', '7e:7b:e1:91:6d:b4',
 'f6:1e:9e:59:30:ab', '16:6e:92:80:6a:03', 'f2:65:14:87:fb:f1',
 '7e:d8:7b:ca:15:af', '42:3a:55:21:a6:98', 'e6:cb:a9:ba:14:55',
 'fa:ca:44:94:4b:77', 'ee:9e:98:8d:71:b6', '26:6d:91:af:f6:43',
 '26:2d:8e:bc:c0:b7', 'fe:ed:15:09:94:0d', 'd2:5d:cd:17:09:9c',
 '46:57:58:ce:7e:d9', 'ee:

In [32]:
attack_test["src_mac"].unique()

<StringArray>
['12:34:01:19:20:f8', '12:34:26:0e:c5:a6', '12:34:8a:36:33:9a',
 '12:34:ae:00:21:04', '12:34:8c:85:5c:fe', 'd6:c6:da:6a:f2:23',
 '12:1c:63:b7:2e:27', 'f2:6f:34:cc:e6:80', '8e:6c:4b:ed:88:52',
 '3a:87:65:11:e9:ef', 'de:f4:23:ef:2e:b2', '12:34:30:40:e0:39',
 '1a:fa:bb:36:54:a1', '5e:08:c6:a3:71:9c', '86:c1:8f:5a:11:6c',
 '5a:3c:29:cd:72:13', 'fa:dc:27:d5:18:ce', '82:8d:f9:07:6e:25',
 '66:55:36:66:da:10', 'ee:8a:8d:b7:9b:70', 'fa:31:32:9a:f0:74',
 '4a:bd:a0:25:00:50', '6a:db:6d:35:cf:81', '36:14:bb:67:13:83',
 'ce:14:54:91:ef:63', '12:34:d1:d1:5e:f0', '12:34:76:cf:4c:7e',
 '12:34:02:8e:4e:30', '12:34:ab:ca:6b:5f', '12:34:95:9d:7a:78',
 '12:34:c3:b2:20:80', '12:34:7e:c2:90:0c', '12:34:2f:bd:b2:c1',
 '12:34:41:50:07:7c', '12:34:73:00:54:80', '12:34:d6:2e:4c:74',
 '12:34:86:5d:b7:96', '12:34:48:bd:51:d3', '12:34:80:b8:c1:2f',
 '12:34:32:bb:6b:ea', '12:34:2c:68:8c:0a', '12:34:b7:47:c9:c9']
Length: 42, dtype: str

In [33]:
mapping_not_found_train=0
for i, row in attack_train.iterrows():
    src_mac=row["src_mac"]
    dst_mac=row["dst_mac"]
    src_ip=row["src_ip"]
    dst_ip=row["dst_ip"]
    if ":" in src_ip and src_mac in mappings:
        attack_train.loc[i, "src_ip"]=mappings[src_mac]  
    elif src_mac not in mappings:
        mapping_not_found_train+=1        
    if ":" in dst_ip and dst_mac in mappings:
        attack_train.loc[i, "dst_ip"]=mappings[dst_mac]
    elif dst_mac not in mappings:
        mapping_not_found_train+=1        

mapping_not_found_test=0
for i, row in attack_test.iterrows():
    src_mac=row["src_mac"]
    dst_mac=row["dst_mac"]
    src_ip=row["src_ip"]
    dst_ip=row["dst_ip"]
    if ":" in src_ip and src_mac in mappings:
        attack_test.loc[i, "src_ip"]=mappings[src_mac] 
    elif src_mac not in mappings:
        mapping_not_found_test+=1
    if ":" in dst_ip and dst_mac in mappings:
        attack_test.loc[i, "dst_ip"]=mappings[dst_mac]
    elif dst_mac not in mappings:
        mapping_not_found_test+=1                

In [34]:
print(mapping_not_found_train)

3516


In [35]:
print(mapping_not_found_test)

2217


In [36]:
attack_train

,id,expiration_id,src_ip,src_mac,src_oui,src_port,dst_ip,dst_mac,dst_oui,dst_port,...,application_is_guessed,application_confidence,requested_server_name,client_fingerprint,server_fingerprint,user_agent,content_type,timestamp,label,step_number
0,318,0,10.0.0.2,12:34:7e:5d:bf:ed,12:34:7e,50784,ff02::1,33:33:00:00:00:01,33:33:00,547,...,0,0,NaN,NaN,NaN,NaN,NaN,2025-08-19 19:14:01.213039+00:00,nmap_10_T5,5
1,176,0,10.0.0.17,12:34:93:88:12:84,12:34:93,0,ff02::2,33:33:00:00:00:02,33:33:00,0,...,0,6,NaN,NaN,NaN,NaN,NaN,2025-08-19 19:14:01.315039+00:00,nmap_10_T5,5
2,177,0,10.0.0.23,12:34:f3:ef:1f:0e,12:34:f3,0,ff02::2,33:33:00:00:00:02,33:33:00,0,...,0,6,NaN,NaN,NaN,NaN,NaN,2025-08-19 19:14:05.411039+00:00,nmap_10_T5,5
3,319,0,10.0.0.2,12:34:7e:5d:bf:ed,12:34:7e,52164,ff02::1,33:33:00:00:00:01,33:33:00,547,...,0,0,NaN,NaN,NaN,NaN,NaN,2025-08-19 19:14:07.247039+00:00,nmap_10_T5,5
4,320,0,10.0.0.2,12:34:7e:5d:bf:ed,12:34:7e,36923,ff02::1,33:33:00:00:00:01,33:33:00,547,...,0,0,NaN,NaN,NaN,NaN,NaN,2025-08-19 19:14:13.284039+00:00,nmap_10_T5,5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
195039,198915,0,10.0.0.2,12:34:7e:5d:bf:ed,12:34:7e,37640,ff02::1,33:33:00:00:00:01,33:33:00,547,...,0,0,NaN,NaN,NaN,NaN,NaN,2025-08-20 05:05:40.292039+00:00,dollar_char,530
195040,198916,0,10.0.0.2,12:34:7e:5d:bf:ed,12:34:7e,48731,ff02::1,33:33:00:00:00:01,33:33:00,547,...,0,0,NaN,NaN,NaN,NaN,NaN,2025-08-20 05:05:47.332039+00:00,dollar_char,530
195041,198917,0,10.0.0.2,12:34:7e:5d:bf:ed,12:34:7e,57248,ff02::1,33:33:00:00:00:01,33:33:00,547,...,0,0,NaN,NaN,NaN,NaN,NaN,2025-08-20 05:05:54.371039+00:00,dollar_char,530
195042,198918,0,10.0.0.2,12:34:7e:5d:bf:ed,12:34:7e,48976,ff02::1,33:33:00:00:00:01,33:33:00,547,...,0,0,NaN,NaN,NaN,NaN,NaN,2025-08-20 05:06:00.411039+00:00,dollar_char,530


In [37]:
attack_test

,id,expiration_id,src_ip,src_mac,src_oui,src_port,dst_ip,dst_mac,dst_oui,dst_port,...,application_is_guessed,application_confidence,requested_server_name,client_fingerprint,server_fingerprint,user_agent,content_type,timestamp,label,step_number
0,296,0,10.0.0.2,12:34:01:19:20:f8,12:34:01,42127,ff02::1,33:33:00:00:00:01,33:33:00,547,...,0,0,NaN,NaN,NaN,NaN,NaN,2025-08-20 11:40:47.375608+00:00,nmap_192_T4,3
1,182,0,10.0.0.7,12:34:26:0e:c5:a6,12:34:26,0,ff02::2,33:33:00:00:00:02,33:33:00,0,...,0,6,NaN,NaN,NaN,NaN,NaN,2025-08-20 11:43:24.644608+00:00,nmap_10_T5,5
2,323,0,10.0.0.2,12:34:01:19:20:f8,12:34:01,40879,ff02::1,33:33:00:00:00:01,33:33:00,547,...,0,0,NaN,NaN,NaN,NaN,NaN,2025-08-20 11:43:26.506608+00:00,nmap_10_T5,5
3,324,0,10.0.0.2,12:34:01:19:20:f8,12:34:01,60562,ff02::1,33:33:00:00:00:01,33:33:00,547,...,0,0,NaN,NaN,NaN,NaN,NaN,2025-08-20 11:43:31.552608+00:00,nmap_10_T5,5
4,183,0,10.0.0.12,12:34:8a:36:33:9a,12:34:8a,0,ff02::2,33:33:00:00:00:02,33:33:00,0,...,0,6,NaN,NaN,NaN,NaN,NaN,2025-08-20 11:43:32.836608+00:00,nmap_10_T5,5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
110271,115201,0,10.0.0.2,12:34:01:19:20:f8,12:34:01,34296,ff02::1,33:33:00:00:00:01,33:33:00,547,...,0,0,NaN,NaN,NaN,NaN,NaN,2025-08-20 21:34:12.816608+00:00,dollar_char,309
110272,115202,0,10.0.0.2,12:34:01:19:20:f8,12:34:01,58492,ff02::1,33:33:00:00:00:01,33:33:00,547,...,0,0,NaN,NaN,NaN,NaN,NaN,2025-08-20 21:34:16.858608+00:00,dollar_char,309
110273,115203,0,10.0.0.2,12:34:01:19:20:f8,12:34:01,52570,ff02::1,33:33:00:00:00:01,33:33:00,547,...,0,0,NaN,NaN,NaN,NaN,NaN,2025-08-20 21:34:23.901608+00:00,dollar_char,309
110274,115204,0,10.0.0.2,12:34:01:19:20:f8,12:34:01,32798,ff02::1,33:33:00:00:00:01,33:33:00,547,...,0,0,NaN,NaN,NaN,NaN,NaN,2025-08-20 21:34:30.948608+00:00,dollar_char,309


In [38]:
mapping_not_found_train=0
for i, row in normal_train.iterrows():
    src_mac=row["src_mac"]
    dst_mac=row["dst_mac"]
    src_ip=row["src_ip"]
    dst_ip=row["dst_ip"]
    if ":" in src_ip and src_mac in mappings:
        normal_train.loc[i, "src_ip"]=mappings[src_mac]    
    elif src_mac not in mappings:
        mapping_not_found_train+=1         
    if ":" in dst_ip and dst_mac in mappings:
        normal_train.loc[i, "dst_ip"]=mappings[dst_mac]
    elif dst_mac not in mappings:
        mapping_not_found_train+=1         

mapping_not_found_test=0
for i, row in normal_test.iterrows():
    src_mac=row["src_mac"]
    dst_mac=row["dst_mac"]
    src_ip=row["src_ip"]
    dst_ip=row["dst_ip"]
    if ":" in src_ip and src_mac in mappings:
        normal_test.loc[i, "src_ip"]=mappings[src_mac]    
    elif src_mac not in mappings:
        mapping_not_found_test+=1        
    if ":" in dst_ip and dst_mac in mappings:
        normal_test.loc[i, "dst_ip"]=mappings[dst_mac]
    elif dst_mac not in mappings:
        mapping_not_found_test+=1         

In [39]:
print(mapping_not_found_train)

9902


In [40]:
print(mapping_not_found_test)

9955


In [41]:
normal_train

,id,expiration_id,src_ip,src_mac,src_oui,src_port,dst_ip,dst_mac,dst_oui,dst_port,...,dst2src_fin_packets,application_name,application_category_name,application_is_guessed,application_confidence,requested_server_name,client_fingerprint,server_fingerprint,user_agent,content_type
0,0,0,::,96:a5:fa:53:10:d0,96:a5:fa,0,ff02::16,33:33:00:00:00:16,33:33:00,0,...,0,ICMPV6,Network,0,6,NaN,NaN,NaN,NaN,NaN
1,1,0,::,fa:be:65:a8:61:93,fa:be:65,0,ff02::1:ffa8:6193,33:33:ff:a8:61:93,33:33:ff,0,...,0,ICMPV6,Network,0,6,NaN,NaN,NaN,NaN,NaN
2,2,0,::,52:1f:a5:ce:9c:13,52:1f:a5,0,ff02::1:ffce:9c13,33:33:ff:ce:9c:13,33:33:ff,0,...,0,ICMPV6,Network,0,6,NaN,NaN,NaN,NaN,NaN
3,3,0,10.0.0.4,12:34:6f:ad:cc:cd,12:34:6f,0,ff02::2,33:33:00:00:00:02,33:33:00,0,...,0,ICMPV6,Network,0,6,NaN,NaN,NaN,NaN,NaN
4,4,0,fe80::4020:90ff:fe67:1afb,42:20:90:67:1a:fb,42:20:90,5353,ff02::fb,33:33:00:00:00:fb,33:33:00,5353,...,0,MDNS,Network,0,6,potassio.local,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9436,9436,0,10.0.0.2,12:34:a5:12:d2:5f,12:34:a5,54632,ff02::1,33:33:00:00:00:01,33:33:00,547,...,0,Unknown,Unspecified,0,0,NaN,NaN,NaN,NaN,NaN
9437,9437,0,10.0.0.2,12:34:a5:12:d2:5f,12:34:a5,33360,ff02::1,33:33:00:00:00:01,33:33:00,547,...,0,Unknown,Unspecified,0,0,NaN,NaN,NaN,NaN,NaN
9438,9438,0,10.0.0.2,12:34:a5:12:d2:5f,12:34:a5,50080,ff02::1,33:33:00:00:00:01,33:33:00,547,...,0,Unknown,Unspecified,0,0,NaN,NaN,NaN,NaN,NaN
9439,9439,0,10.0.0.2,12:34:a5:12:d2:5f,12:34:a5,46885,ff02::1,33:33:00:00:00:01,33:33:00,547,...,0,Unknown,Unspecified,0,0,NaN,NaN,NaN,NaN,NaN


In [42]:
normal_test

,id,expiration_id,src_ip,src_mac,src_oui,src_port,dst_ip,dst_mac,dst_oui,dst_port,...,dst2src_fin_packets,application_name,application_category_name,application_is_guessed,application_confidence,requested_server_name,client_fingerprint,server_fingerprint,user_agent,content_type
0,0,0,::,96:a5:fa:53:10:d0,96:a5:fa,0,ff02::16,33:33:00:00:00:16,33:33:00,0,...,0,ICMPV6,Network,0,6,NaN,NaN,NaN,NaN,NaN
1,1,0,::,62:6a:e0:b0:96:9f,62:6a:e0,0,ff02::1:ffb0:969f,33:33:ff:b0:96:9f,33:33:ff,0,...,0,ICMPV6,Network,0,6,NaN,NaN,NaN,NaN,NaN
2,2,0,::,fe:73:58:53:d6:db,fe:73:58,0,ff02::1:ff53:d6db,33:33:ff:53:d6:db,33:33:ff,0,...,0,ICMPV6,Network,0,6,NaN,NaN,NaN,NaN,NaN
3,3,0,::,1a:e0:de:ee:df:90,1a:e0:de,0,ff02::1:ffee:df90,33:33:ff:ee:df:90,33:33:ff,0,...,0,ICMPV6,Network,0,6,NaN,NaN,NaN,NaN,NaN
4,4,0,::,c6:c5:16:b7:17:bb,c6:c5:16,0,ff02::1:ffb7:17bb,33:33:ff:b7:17:bb,33:33:ff,0,...,0,ICMPV6,Network,0,6,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9406,9406,0,10.0.0.2,12:34:7f:ca:93:21,12:34:7f,49520,ff02::1,33:33:00:00:00:01,33:33:00,547,...,0,Unknown,Unspecified,0,0,NaN,NaN,NaN,NaN,NaN
9407,9407,0,10.0.0.2,12:34:7f:ca:93:21,12:34:7f,37752,ff02::1,33:33:00:00:00:01,33:33:00,547,...,0,Unknown,Unspecified,0,0,NaN,NaN,NaN,NaN,NaN
9408,9408,0,10.0.0.2,12:34:7f:ca:93:21,12:34:7f,59022,ff02::1,33:33:00:00:00:01,33:33:00,547,...,0,Unknown,Unspecified,0,0,NaN,NaN,NaN,NaN,NaN
9409,9409,0,10.0.0.2,12:34:7f:ca:93:21,12:34:7f,43369,ff02::1,33:33:00:00:00:01,33:33:00,547,...,0,Unknown,Unspecified,0,0,NaN,NaN,NaN,NaN,NaN


In [43]:
attack_test=attack_test[attack_test["src_ip"].str.contains(".")&attack_test["dst_ip"].str.contains(".", regex=False)]
attack_train=attack_train[attack_train["src_ip"].str.contains(".")&attack_train["dst_ip"].str.contains(".", regex=False)]

In [44]:
normal_test=normal_test[normal_test["src_ip"].str.contains(".")&normal_test["dst_ip"].str.contains(".", regex=False)]
normal_train=normal_train[normal_train["src_ip"].str.contains(".")&normal_train["dst_ip"].str.contains(".", regex=False)]

In [45]:
attack_test.info()

<class 'pandas.DataFrame'>
Index: 108132 entries, 36 to 110270
Data columns (total 89 columns):
 #   Column                        Non-Null Count   Dtype  
---  ------                        --------------   -----  
 0   id                            108132 non-null  int64  
 1   expiration_id                 108132 non-null  int64  
 2   src_ip                        108132 non-null  str    
 3   src_mac                       108132 non-null  str    
 4   src_oui                       108132 non-null  str    
 5   src_port                      108132 non-null  int64  
 6   dst_ip                        108132 non-null  str    
 7   dst_mac                       108132 non-null  str    
 8   dst_oui                       108132 non-null  str    
 9   dst_port                      108132 non-null  int64  
 10  protocol                      108132 non-null  int64  
 11  ip_version                    108132 non-null  int64  
 12  vlan_id                       108132 non-null  int64  
 13 

In [46]:
attack_test["src_ip"].value_counts()

src_ip
10.0.0.2     105568
10.0.0.15      1342
10.0.0.20       108
10.0.0.18       103
10.0.0.19       103
10.0.0.12        82
10.0.0.13        82
10.0.0.14        82
10.0.0.16        82
10.0.0.17        82
10.0.0.21        46
10.0.0.22        46
10.0.0.23        46
10.0.0.4         40
10.0.0.5         40
10.0.0.6         40
10.0.0.7         40
10.0.0.8         40
10.0.0.9         40
10.0.0.10        40
10.0.0.11        40
10.0.0.1         40
Name: count, dtype: int64

In [47]:
attack_test["dst_ip"].value_counts()

dst_ip
10.0.0.15    5858
10.0.0.22    5526
10.0.0.1     5266
10.0.0.20    5080
10.0.0.19    5079
10.0.0.5     5078
10.0.0.4     5078
10.0.0.6     5078
10.0.0.7     5078
10.0.0.8     5078
10.0.0.9     5078
10.0.0.10    5078
10.0.0.11    5078
10.0.0.16    5078
10.0.0.18    5078
10.0.0.17    5078
10.0.0.23    5078
10.0.0.21    5078
10.0.0.13    5077
10.0.0.12    5077
10.0.0.14    5077
10.0.0.2       78
Name: count, dtype: int64

In [48]:
attack_train.info()

<class 'pandas.DataFrame'>
Index: 191809 entries, 16 to 195035
Data columns (total 89 columns):
 #   Column                        Non-Null Count   Dtype  
---  ------                        --------------   -----  
 0   id                            191809 non-null  int64  
 1   expiration_id                 191809 non-null  int64  
 2   src_ip                        191809 non-null  str    
 3   src_mac                       191809 non-null  str    
 4   src_oui                       191809 non-null  str    
 5   src_port                      191809 non-null  int64  
 6   dst_ip                        191809 non-null  str    
 7   dst_mac                       191809 non-null  str    
 8   dst_oui                       191809 non-null  str    
 9   dst_port                      191809 non-null  int64  
 10  protocol                      191809 non-null  int64  
 11  ip_version                    191809 non-null  int64  
 12  vlan_id                       191809 non-null  int64  
 13 

In [49]:
attack_train["src_ip"].value_counts()

src_ip
10.0.0.2     189543
10.0.0.15       995
10.0.0.20       289
10.0.0.19       114
10.0.0.18       113
10.0.0.17       113
10.0.0.16       113
10.0.0.14       113
10.0.0.21        36
10.0.0.22        36
10.0.0.23        36
10.0.0.4         28
10.0.0.5         28
10.0.0.6         28
10.0.0.7         28
10.0.0.8         28
10.0.0.9         28
10.0.0.10        28
10.0.0.11        28
10.0.0.12        28
10.0.0.1         28
10.0.0.13        28
Name: count, dtype: int64

In [50]:
attack_train["dst_ip"].value_counts()

dst_ip
10.0.0.15    9613
10.0.0.21    9528
10.0.0.1     9355
10.0.0.4     9070
10.0.0.6     9070
10.0.0.8     9070
10.0.0.5     9070
10.0.0.7     9070
10.0.0.13    9070
10.0.0.12    9070
10.0.0.11    9070
10.0.0.9     9070
10.0.0.10    9070
10.0.0.23    9070
10.0.0.22    9070
10.0.0.20    9068
10.0.0.16    9067
10.0.0.19    9067
10.0.0.18    9067
10.0.0.17    9067
10.0.0.14    9067
10.0.0.2       70
Name: count, dtype: int64

In [51]:
normal_test.info()

<class 'pandas.DataFrame'>
Index: 376 entries, 5 to 9159
Data columns (total 86 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   id                            376 non-null    int64  
 1   expiration_id                 376 non-null    int64  
 2   src_ip                        376 non-null    str    
 3   src_mac                       376 non-null    str    
 4   src_oui                       376 non-null    str    
 5   src_port                      376 non-null    int64  
 6   dst_ip                        376 non-null    str    
 7   dst_mac                       376 non-null    str    
 8   dst_oui                       376 non-null    str    
 9   dst_port                      376 non-null    int64  
 10  protocol                      376 non-null    int64  
 11  ip_version                    376 non-null    int64  
 12  vlan_id                       376 non-null    int64  
 13  tunnel_id           

In [52]:
normal_test["src_ip"].value_counts()

src_ip
10.0.0.2     50
10.0.0.4     28
10.0.0.5     28
10.0.0.7     27
10.0.0.8     27
10.0.0.9     27
10.0.0.12    27
10.0.0.15    27
10.0.0.11    27
10.0.0.13    27
10.0.0.16    27
10.0.0.17    27
10.0.0.19    27
Name: count, dtype: int64

In [53]:
normal_test["dst_ip"].value_counts()

dst_ip
10.0.0.1     326
10.0.0.11      5
10.0.0.23      4
10.0.0.12      4
10.0.0.7       4
10.0.0.8       4
10.0.0.15      3
10.0.0.4       3
10.0.0.6       3
10.0.0.5       3
10.0.0.21      2
10.0.0.16      2
10.0.0.20      2
10.0.0.9       2
10.0.0.14      2
10.0.0.22      2
10.0.0.18      1
10.0.0.10      1
10.0.0.13      1
10.0.0.19      1
10.0.0.17      1
Name: count, dtype: int64

In [54]:
normal_train.info()

<class 'pandas.DataFrame'>
Index: 443 entries, 194 to 9181
Data columns (total 86 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   id                            443 non-null    int64  
 1   expiration_id                 443 non-null    int64  
 2   src_ip                        443 non-null    str    
 3   src_mac                       443 non-null    str    
 4   src_oui                       443 non-null    str    
 5   src_port                      443 non-null    int64  
 6   dst_ip                        443 non-null    str    
 7   dst_mac                       443 non-null    str    
 8   dst_oui                       443 non-null    str    
 9   dst_port                      443 non-null    int64  
 10  protocol                      443 non-null    int64  
 11  ip_version                    443 non-null    int64  
 12  vlan_id                       443 non-null    int64  
 13  tunnel_id         

In [55]:
normal_train["src_ip"].value_counts()

src_ip
10.0.0.2     61
10.0.0.4     27
10.0.0.5     27
10.0.0.7     27
10.0.0.8     27
10.0.0.9     27
10.0.0.15    27
10.0.0.13    27
10.0.0.11    27
10.0.0.12    27
10.0.0.16    27
10.0.0.17    27
10.0.0.19    27
10.0.0.22    26
10.0.0.23    26
10.0.0.6      2
10.0.0.10     2
10.0.0.14     2
Name: count, dtype: int64

In [56]:
normal_train["dst_ip"].value_counts()

dst_ip
10.0.0.1     382
10.0.0.22      5
10.0.0.12      5
10.0.0.17      5
10.0.0.4       5
10.0.0.14      5
10.0.0.18      4
10.0.0.13      3
10.0.0.16      3
10.0.0.23      3
10.0.0.6       3
10.0.0.10      3
10.0.0.5       3
10.0.0.21      3
10.0.0.8       3
10.0.0.20      3
10.0.0.9       2
10.0.0.11      1
10.0.0.7       1
10.0.0.15      1
Name: count, dtype: int64

In [57]:
attack_steps = (
    attack_train
    .groupby('label', as_index=False)
    .agg(total_steps=('step_number', 'nunique'))
)

print(attack_steps)

                   label  total_steps
0  brute_force_malformed            1
1            dollar_char          166
2             nmap_10_T5            9
3            nmap_banner           14
4              nmap_mqtt           21
5               nmap_sub            7
6               scp_inst            7


In [58]:
attack_steps = (
    attack_test
    .groupby('label', as_index=False)
    .agg(total_steps=('step_number', 'nunique'))
)

print(attack_steps)

                   label  total_steps
0  brute_force_malformed            1
1            dollar_char           63
2               mqtt_cat            1
3             nmap_10_T5            5
4            nmap_banner           20
5              nmap_mqtt           29
6               scp_inst            9


## Augment normal dataset

In [59]:
keywords_excluded=["id", "ip", "mac", "oui", "first_seen", "last_seen", "duration", "fingerprint", "port"]

In [60]:
def generate_dataframe(df, start_time, repetitions):
    
    # deals with time features
    delta=np.random.normal(0,500,(df.shape[0],1)).round().astype(int)
    df=df.sort_values(by="bidirectional_first_seen_ms")
    n_df=df.copy()
    cols=["bidirectional_first_seen_ms","bidirectional_last_seen_ms"]
    n_df[cols]=df[cols].to_numpy() // repetitions + delta + start_time
    n_df["bidirectional_duration_ms"]=n_df["bidirectional_last_seen_ms"]-n_df["bidirectional_first_seen_ms"]

    # deals with numerical features
    num_cols = [col for col in n_df.columns if n_df[col].nunique()>2**7 and all([keyword not in col for keyword in keywords_excluded])]
    f_cols= [col for col in num_cols if np.issubdtype(n_df[col].dtype, np.floating)]
    i_cols= [col for col in num_cols if np.issubdtype(n_df[col].dtype, np.integer)]

    f_noise = np.random.normal(np.zeros(len(f_cols)), 0.1 * n_df[f_cols].std(), (n_df.shape[0],len(f_cols)))
    i_noise = np.round(np.random.normal(np.zeros(len(i_cols)), 0.1 * n_df[i_cols].std(), (n_df.shape[0],len(i_cols))))

    n_df[f_cols] = n_df[f_cols] + f_noise
    n_df[i_cols] = (n_df[i_cols] + i_noise).astype(int)

    # deals with categorical features
    cat_cols=[col for col in n_df.columns if col not in num_cols and all([keyword not in col for keyword in keywords_excluded])]
    n_df[cat_cols]=pd.DataFrame({col: np.random.choice(n_df[col], size=n_df.shape[0]) for col in cat_cols})
    
    return n_df

In [61]:
dfs=[normal_train]
for i in range(1, 500):
    dfs.append(generate_dataframe(dfs[0], dfs[i-1].iloc[-1]["bidirectional_last_seen_ms"], 500))

In [62]:
normal_train=pd.concat(dfs, ignore_index=True)
normal_train.info()

<class 'pandas.DataFrame'>
RangeIndex: 221500 entries, 0 to 221499
Data columns (total 86 columns):
 #   Column                        Non-Null Count   Dtype  
---  ------                        --------------   -----  
 0   id                            221500 non-null  int64  
 1   expiration_id                 221500 non-null  int64  
 2   src_ip                        221500 non-null  str    
 3   src_mac                       221500 non-null  str    
 4   src_oui                       221500 non-null  str    
 5   src_port                      221500 non-null  int64  
 6   dst_ip                        221500 non-null  str    
 7   dst_mac                       221500 non-null  str    
 8   dst_oui                       221500 non-null  str    
 9   dst_port                      221500 non-null  int64  
 10  protocol                      16411 non-null   float64
 11  ip_version                    221500 non-null  int64  
 12  vlan_id                       221500 non-null  int64  


In [63]:
normal_train

,id,expiration_id,src_ip,src_mac,src_oui,src_port,dst_ip,dst_mac,dst_oui,dst_port,...,dst2src_fin_packets,application_name,application_category_name,application_is_guessed,application_confidence,requested_server_name,client_fingerprint,server_fingerprint,user_agent,content_type
0,194,1,10.0.0.22,12:34:73:b3:6d:24,12:34:73,34863,10.0.0.1,12:34:a5:15:3c:3e,12:34:a5,1883,...,0.0,MQTT,RPC,0.0,6,NaN,NaN,NaN,NaN,NaN
1,201,1,10.0.0.23,12:34:d4:3b:81:a1,12:34:d4,50323,10.0.0.1,12:34:a5:15:3c:3e,12:34:a5,1883,...,0.0,MQTT,RPC,0.0,6,NaN,NaN,NaN,NaN,NaN
2,246,0,10.0.0.4,12:34:6f:ad:cc:cd,12:34:6f,57437,10.0.0.1,12:34:a5:15:3c:3e,12:34:a5,1883,...,1.0,MQTT,RPC,1.0,1,NaN,NaN,NaN,NaN,NaN
3,248,0,10.0.0.5,12:34:d7:7e:03:0b,12:34:d7,53963,10.0.0.1,12:34:a5:15:3c:3e,12:34:a5,1883,...,1.0,MQTT,RPC,1.0,1,NaN,NaN,NaN,NaN,NaN
4,249,0,10.0.0.6,12:34:7f:bb:40:6c,12:34:7f,50217,10.0.0.1,12:34:a5:15:3c:3e,12:34:a5,1883,...,1.0,MQTT,RPC,1.0,1,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
221495,9150,0,10.0.0.7,12:34:4b:9c:8f:9a,12:34:4b,56729,10.0.0.1,12:34:a5:15:3c:3e,12:34:a5,1883,...,NaN,NaN,NaN,NaN,6,NaN,NaN,NaN,NaN,NaN
221496,9154,0,10.0.0.13,12:34:0d:f0:9c:36,12:34:0d,43077,10.0.0.1,12:34:a5:15:3c:3e,12:34:a5,1883,...,NaN,NaN,NaN,NaN,6,NaN,NaN,NaN,NaN,NaN
221497,9167,0,10.0.0.11,12:34:a1:83:ca:33,12:34:a1,58305,10.0.0.1,12:34:a5:15:3c:3e,12:34:a5,1883,...,NaN,NaN,NaN,NaN,6,NaN,NaN,NaN,NaN,NaN
221498,9174,0,10.0.0.16,12:34:9f:d5:63:9f,12:34:9f,44161,10.0.0.1,12:34:a5:15:3c:3e,12:34:a5,1883,...,NaN,NaN,NaN,NaN,6,NaN,NaN,NaN,NaN,NaN


In [64]:
dfs=[normal_test]
for i in range(1, 500):
    dfs.append(generate_dataframe(dfs[0], dfs[i-1].iloc[-1]["bidirectional_last_seen_ms"], 500))

In [65]:
normal_test=pd.concat(dfs, ignore_index=True)
normal_test.info()

<class 'pandas.DataFrame'>
RangeIndex: 188000 entries, 0 to 187999
Data columns (total 86 columns):
 #   Column                        Non-Null Count   Dtype  
---  ------                        --------------   -----  
 0   id                            188000 non-null  int64  
 1   expiration_id                 188000 non-null  int64  
 2   src_ip                        188000 non-null  str    
 3   src_mac                       188000 non-null  str    
 4   src_oui                       188000 non-null  str    
 5   src_port                      188000 non-null  int64  
 6   dst_ip                        188000 non-null  str    
 7   dst_mac                       188000 non-null  str    
 8   dst_oui                       188000 non-null  str    
 9   dst_port                      188000 non-null  int64  
 10  protocol                      13350 non-null   float64
 11  ip_version                    188000 non-null  int64  
 12  vlan_id                       188000 non-null  int64  


In [66]:
normal_test

,id,expiration_id,src_ip,src_mac,src_oui,src_port,dst_ip,dst_mac,dst_oui,dst_port,...,dst2src_fin_packets,application_name,application_category_name,application_is_guessed,application_confidence,requested_server_name,client_fingerprint,server_fingerprint,user_agent,content_type
0,5,0,10.0.0.7,12:34:e6:e8:7c:cc,12:34:e6,49695,10.0.0.1,12:34:4c:3c:6b:02,12:34:4c,1883,...,1.0,MQTT,RPC,1.0,1,NaN,NaN,NaN,NaN,NaN
1,39,0,10.0.0.8,12:34:c3:58:c9:90,12:34:c3,37429,10.0.0.1,12:34:4c:3c:6b:02,12:34:4c,1883,...,1.0,MQTT,RPC,1.0,1,NaN,NaN,NaN,NaN,NaN
2,46,0,10.0.0.4,12:34:da:e9:f7:5f,12:34:da,47577,10.0.0.1,12:34:4c:3c:6b:02,12:34:4c,1883,...,1.0,MQTT,RPC,1.0,1,NaN,NaN,NaN,NaN,NaN
3,47,0,10.0.0.4,12:34:da:e9:f7:5f,12:34:da,35271,10.0.0.1,12:34:4c:3c:6b:02,12:34:4c,1883,...,1.0,MQTT,RPC,0.0,6,NaN,NaN,NaN,NaN,NaN
4,51,0,10.0.0.5,12:34:fa:fb:5e:7d,12:34:fa,47799,10.0.0.1,12:34:4c:3c:6b:02,12:34:4c,1883,...,1.0,MQTT,RPC,1.0,1,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
187995,9131,0,10.0.0.5,12:34:fa:fb:5e:7d,12:34:fa,60129,10.0.0.1,12:34:4c:3c:6b:02,12:34:4c,1883,...,NaN,NaN,NaN,NaN,6,NaN,NaN,NaN,NaN,NaN
187996,9135,0,10.0.0.13,12:34:ee:3f:53:27,12:34:ee,53493,10.0.0.1,12:34:4c:3c:6b:02,12:34:4c,1883,...,NaN,NaN,NaN,NaN,6,NaN,NaN,NaN,NaN,NaN
187997,9149,0,10.0.0.11,12:34:43:31:d1:bc,12:34:43,35049,10.0.0.1,12:34:4c:3c:6b:02,12:34:4c,1883,...,NaN,NaN,NaN,NaN,6,NaN,NaN,NaN,NaN,NaN
187998,9154,0,10.0.0.16,12:34:32:47:fc:69,12:34:32,36113,10.0.0.1,12:34:4c:3c:6b:02,12:34:4c,1883,...,NaN,NaN,NaN,NaN,6,NaN,NaN,NaN,NaN,NaN


## Convert to absolute time

In [67]:
from datetime import datetime, timedelta
def convert_to_absolute_time(epoch_seconds, start_time):
    utc=pytz.UTC
    relative_time = timedelta(milliseconds=epoch_seconds)
    absolute_time = start_time + relative_time
    absolute_time = utc.localize(absolute_time)
    return absolute_time

In [68]:
import pytz
start_time = np.load("./data/dollar_char_train/dollar_char_train.npz", allow_pickle=True)
start_time = start_time['datetime']
start_time = start_time.item().replace(hour=start_time.item().hour -2)

In [69]:
normal_train['timestamp'] = normal_train['bidirectional_first_seen_ms'].apply(convert_to_absolute_time, start_time=start_time)
normal_train["label"] = "normal"
normal_train["step_number"] = -1

In [70]:
normal_test['timestamp'] = normal_test['bidirectional_first_seen_ms'].apply(convert_to_absolute_time, start_time=start_time)
normal_test["label"] = "normal"
normal_test["step_number"] = -1

## Save augmented normal dataset

In [71]:
normal_train.to_csv("data/normal_train.csv", index=False)
normal_test.to_csv("data/normal_test.csv", index=False)

## Concatenate the dataframes

In [72]:
df_test=pd.concat([normal_test, attack_test], ignore_index=True)

In [73]:
df_test.info()

<class 'pandas.DataFrame'>
RangeIndex: 296132 entries, 0 to 296131
Data columns (total 89 columns):
 #   Column                        Non-Null Count   Dtype  
---  ------                        --------------   -----  
 0   id                            296132 non-null  int64  
 1   expiration_id                 296132 non-null  int64  
 2   src_ip                        296132 non-null  str    
 3   src_mac                       296132 non-null  str    
 4   src_oui                       296132 non-null  str    
 5   src_port                      296132 non-null  int64  
 6   dst_ip                        296132 non-null  str    
 7   dst_mac                       296132 non-null  str    
 8   dst_oui                       296132 non-null  str    
 9   dst_port                      296132 non-null  int64  
 10  protocol                      121482 non-null  float64
 11  ip_version                    296132 non-null  int64  
 12  vlan_id                       296132 non-null  int64  


In [74]:
df_train=pd.concat([normal_train, attack_train], ignore_index=True)

In [75]:
df_train.info()

<class 'pandas.DataFrame'>
RangeIndex: 413309 entries, 0 to 413308
Data columns (total 89 columns):
 #   Column                        Non-Null Count   Dtype  
---  ------                        --------------   -----  
 0   id                            413309 non-null  int64  
 1   expiration_id                 413309 non-null  int64  
 2   src_ip                        413309 non-null  str    
 3   src_mac                       413309 non-null  str    
 4   src_oui                       413309 non-null  str    
 5   src_port                      413309 non-null  int64  
 6   dst_ip                        413309 non-null  str    
 7   dst_mac                       413309 non-null  str    
 8   dst_oui                       413309 non-null  str    
 9   dst_port                      413309 non-null  int64  
 10  protocol                      208220 non-null  float64
 11  ip_version                    413309 non-null  int64  
 12  vlan_id                       413309 non-null  int64  


## Save results

In [76]:
df_train.to_csv("data/train.csv", index=False)
df_test.to_csv("data/test.csv", index=False)